# Racing Game - Map Tour
This notebook loads the racing bitstream and smoothly pans the camera around the entire 20×15 tile map (1280×960 pixels).  
The visible screen is 640×480, so the camera can scroll from (0,0) to (640,480).

In [3]:
from pynq import Overlay
import time

# Load the bitstream
overlay = Overlay('top_design.bit')
regs = overlay.racing_renderer_0.mmio

print('Bitstream loaded!')
print(f'Overlay IP blocks: {overlay.ip_dict.keys()}')

Bitstream loaded!
Overlay IP blocks: dict_keys(['physics_axi_ip_0', 'racing_renderer_0', 'processing_system7_0'])


In [2]:
print(f'Overlay IP blocks: {overlay.ip_dict.keys()}')

Overlay IP blocks: dict_keys(['physics_axi_ip_0', 'racing_renderer_0', 'processing_system7_0'])


In [4]:
# Register map:
#   0x00: cam_x   (11 bits, 0-2047)
#   0x04: cam_y   (10 bits, 0-1023)
#   0x08: car1_x  (11 bits)
#   0x0C: car1_y  (10 bits)
#   0x10: car2_x  (11 bits)
#   0x14: car2_y  (10 bits)

def set_camera(x, y):
    """Set camera position (top-left corner of visible screen)"""
    regs.write(0x00, int(x))
    regs.write(0x04, int(y))

def set_car1(x, y):
    """Set car 1 position (world coordinates)"""
    regs.write(0x08, int(x))
    regs.write(0x0C, int(y))

def set_car2(x, y):
    """Set car 2 position (world coordinates)"""
    regs.write(0x10, int(x))
    regs.write(0x14, int(y))

print('Helper functions ready.')

Helper functions ready.


In [5]:
# Map dimensions
WORLD_W = 20 * 64   # 1280 pixels
WORLD_H = 15 * 64   # 960 pixels
SCREEN_W = 640
SCREEN_H = 480

# Maximum camera positions (so we don't scroll past the map edge)
MAX_CAM_X = WORLD_W - SCREEN_W   # 640
MAX_CAM_Y = WORLD_H - SCREEN_H   # 480

print(f'World: {WORLD_W}x{WORLD_H}')
print(f'Screen: {SCREEN_W}x{SCREEN_H}')
print(f'Camera range: (0,0) to ({MAX_CAM_X},{MAX_CAM_Y})')

World: 1280x960
Screen: 640x480
Camera range: (0,0) to (640,480)


In [6]:
# Hide the cars off-screen so they don't obstruct the view
set_car1(0, 0)
set_car2(0, 0)

# Start at top-left
set_camera(0, 0)
print('Camera at (0,0) - top left corner')
time.sleep(1)

Camera at (0,0) - top left corner


In [8]:
# === SMOOTH MAP TOUR ===
# Pans around the entire map border, then does a diagonal sweep.
#
# Route: top-left -> top-right -> bottom-right -> bottom-left -> top-left -> diagonal
#
# Adjust SPEED to make it faster/slower (pixels per step)
# Adjust DELAY to control smoothness (seconds between steps)

SPEED = 2        # pixels per step
DELAY = 0.016    # ~60fps

def smooth_pan(start_x, start_y, end_x, end_y, speed=SPEED, delay=DELAY):
    """Smoothly pan camera from one position to another."""
    dx = end_x - start_x
    dy = end_y - start_y
    dist = max(abs(dx), abs(dy))
    if dist == 0:
        return
    steps = max(1, dist // speed)
    
    for i in range(steps + 1):
        t = i / steps
        x = int(start_x + dx * t)
        y = int(start_y + dy * t)
        # Clamp to valid range
        x = max(0, min(MAX_CAM_X, x))
        y = max(0, min(MAX_CAM_Y, y))
        set_camera(x, y)
        time.sleep(delay)

print('Starting map tour...')
print()

# Top-left to top-right
print('Panning right along top edge...')
smooth_pan(0, 0, MAX_CAM_X, 0)
time.sleep(0.5)

# Top-right to bottom-right
print('Panning down along right edge...')
smooth_pan(MAX_CAM_X, 0, MAX_CAM_X, MAX_CAM_Y)
time.sleep(0.5)

# Bottom-right to bottom-left
print('Panning left along bottom edge...')
smooth_pan(MAX_CAM_X, MAX_CAM_Y, 0, MAX_CAM_Y)
time.sleep(0.5)

# Bottom-left to top-left
print('Panning up along left edge...')
smooth_pan(0, MAX_CAM_Y, 0, 0)
time.sleep(0.5)

# Diagonal sweep: top-left to bottom-right
print('Diagonal sweep...')
smooth_pan(0, 0, MAX_CAM_X, MAX_CAM_Y)
time.sleep(0.5)

# Back to center
center_x = MAX_CAM_X // 2
center_y = MAX_CAM_Y // 2
print(f'Returning to center ({center_x}, {center_y})...')
smooth_pan(MAX_CAM_X, MAX_CAM_Y, center_x, center_y)

print()
print('Tour complete!')

Starting map tour...

Panning right along top edge...
Panning down along right edge...
Panning left along bottom edge...
Panning up along left edge...
Diagonal sweep...
Returning to center (320, 240)...

Tour complete!


In [ ]:
# === MANUAL CAMERA CONTROL ===
# Use this cell to jump to any position on the map.

# Set camera to a specific tile (tile coordinates, not pixels)
def camera_to_tile(tile_col, tile_row):
    """Move camera so the given tile is near the top-left of screen."""
    x = min(tile_col * 64, MAX_CAM_X)
    y = min(tile_row * 64, MAX_CAM_Y)
    set_camera(x, y)
    print(f'Camera at tile ({tile_col},{tile_row}) = pixel ({x},{y})')

# Examples:
# camera_to_tile(0, 0)    # top-left of map
# camera_to_tile(10, 7)   # center of map
# camera_to_tile(19, 14)  # bottom-right of map

camera_to_tile(0, 0)

In [ ]:
# === SLOW CRAWL - SEE EVERY TILE ===
# Scans row by row across the entire map, pausing at each tile.
# This lets you inspect every tile on the map.

PAUSE_PER_TILE = 0.3  # seconds to pause at each tile position
PAN_SPEED = 4         # pixels per step when moving between tiles

# How many tile positions we can show (limited by camera range)
cam_tile_cols = MAX_CAM_X // 64 + 1  # number of distinct column positions
cam_tile_rows = MAX_CAM_Y // 64 + 1  # number of distinct row positions

print(f'Scanning {cam_tile_cols} x {cam_tile_rows} camera positions...')
print('Press Ctrl+C (interrupt kernel) to stop early.')
print()

try:
    for row in range(cam_tile_rows):
        for col in range(cam_tile_cols):
            x = min(col * 64, MAX_CAM_X)
            y = min(row * 64, MAX_CAM_Y)
            set_camera(x, y)
            time.sleep(PAUSE_PER_TILE)
        print(f'  Row {row} done')
except KeyboardInterrupt:
    print('\nStopped.')

print('Scan complete!')